# The RAG Spectrum: Exploring 7 Distinct Architectures from Naive to Agentic

Companion notebook for [the complete To Data & Beyond tutorial](https://todatabeyond.com/blog/the-rag-spectrum-exploring-7-distinct-architectures-from-naive-to-agentic). View the [maintained notebook on GitHub](https://github.com/To-Data-Beyond/Generative-AI-Techanical-Tutorials/blob/main/RAG_Spectrum_Architectures.ipynb).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/To-Data-Beyond/Generative-AI-Techanical-Tutorials/blob/main/RAG_Spectrum_Architectures.ipynb)


## Before you begin

The article examples illustrate distinct RAG patterns. Add your own documents, credentials, and provider configuration before running them end to end.

This notebook intentionally contains no saved execution outputs or credentials.


## Environment setup


In [ ]:
!pip install -q langchain langchain-community langchain-openai faiss-cpu sentence-transformers


Mastering RAG: 7 Key Architectures to Elevate Your LLM Applications

Retrieval Augmented Generation (RAG) has emerged as a pivotal technique for enhancing Large Language Models (LLMs) by grounding their responses in external, verifiable knowledge.

However, the RAG landscape is diverse, with various architectural patterns suiting different needs and data types. This article provides a comprehensive overview of seven key RAG architectures, starting from foundational “Naive RAG” and progressing through more sophisticated models like “Retrieve-and-Rerank,” “Multimodal RAG” for handling diverse data, “Graph RAG” leveraging structured knowledge, and “Hybrid RAG” blending techniques.

It further explores advanced “Agentic RAG” systems, including “Router RAG,” where an agent decides retrieval strategy, and “Multi-Agent RAG,” enabling collaborative problem-solving.

For each architecture, we discuss its core principles, illustrate its workflow (often with conceptual code examples), and highlight its specific advantages and ideal use cases, empowering readers to select and implement the most effective RAG strategy for their applications.

## Table of Contents

---

## 1. Naive RAG: Basic Retrieval + Generation

Naive RAG is the foundational architecture of Retrieval-Augmented Generation systems. It involves three core steps: retrieval, augmentation, and generation. First, documents are chunked and passed through an embedding model to create vector representations. These are stored in a vector database. When a user submits a query, it’s converted into a vector and matched against the document vectors to retrieve the most relevant chunks.

Next, the retrieved chunks (context) are combined with the query in a prompt template. This augmented prompt is passed to a large language model (LLM), which then generates a response. The strength of Naive RAG lies in its simplicity and efficiency, but it lacks any form of result re-ranking or complex decision-making, which can impact the accuracy of answers when documents are noisy or loosely related.

Here’s a simple example using LangChain and FAISS:


In [ ]:
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import TextLoader

# Load and split documents
loader = TextLoader("data.txt")
documents = loader.load()

# Create vector store
embedding_model = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(documents, embedding_model)

# Initialize retriever and LLM
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(model_name="gpt-3.5-turbo")

# Setup RAG pipeline
rag_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

# Ask a question
response = rag_chain.run("What is RAG?")
print(response)


This code sets up a simple RAG pipeline: load documents, embed and store them, retrieve relevant chunks based on a query, and generate a response using an LLM. No re-ranking or advanced reasoning is involved — just plain retrieval and generation.

---

## 2. Retrieve-and-Rerank RAG: Smarter Retrieval via Ranking

This architecture enhances basic RAG by adding an intelligent reranking step to improve the quality of retrieved context:

1. Embedding & Vector Retrieval (Top-K): Like Naive RAG, the process begins by embedding the user query and comparing it to document embeddings stored in a vector database. This initial step retrieves the top k=25 most similar chunks based on vector similarity.
2. Reranker Model (Relevance Scoring): Instead of immediately sending these top-25 results to the LLM, we pass them through a reranker — a separate model trained to evaluate the semantic relevance of query-passage pairs more precisely (e.g., using models like bge-reranker or colBERT). This model scores each of the 25 passages and ranks them based on actual relevance (not just vector similarity).
3. Final Selection (Top-n): From these reranked results, only the top n=4 passages are selected to build the prompt for the LLM.
4. LLM Generation: The final, highly relevant context is sent to the language model, improving both precision and faithfulness of the answer.

### 2.1. Why it’s Smarter than Naive RAG?

- Reduces irrelevant noise from the vector store by not trusting vector similarity alone.
- Adds a second layer of semantic filtering, ensuring only the most relevant information reaches the LLM.
- Especially useful when your corpus is large or noisy, or when embeddings are coarse.

Here is a LangChain Pseudocode:


In [ ]:
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank

# Create vector DB from docs
vectorstore = FAISS.from_documents(docs, HuggingFaceEmbeddings())

# Use a reranker as document compressor
reranker = CohereRerank(model="rerank-english-v2.0", top_n=4)

# Wrap retriever with reranker
retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=vectorstore.as_retriever(search_kwargs={"k": 25})
)

# Setup RetrievalQA pipeline
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(), retriever=retriever
)

response = qa_chain.run("What is Retrieve-and-Rerank RAG?")
print(response)


---

## 3. Multimodal RAG: Handles various data types (text, image)

Multimodal RAG extends the capabilities of traditional RAG systems to understand and process information from various data types beyond just text. This includes images, videos, audio clips, and potentially other sensory data.

The core idea is to leverage multimodal embedding models that can represent different data modalities (e.g., an image and its textual description, or a video frame and spoken words) in a shared semantic vector space.

This allows for cross-modal retrieval, where a query in one modality (e.g., a text question) can retrieve relevant information from another modality (e.g., an image or a video segment).

The architecture, as shown in the diagram, typically involves an ingestion pipeline where “Enterprise Data” like “Videos” and “Images” undergoes “Ingest / Data Preprocessing.”

This preprocessed data is then fed into a “Multimodal Embedding Model” to generate vector representations, which are stored in an “Index / Vector Database.” When a “User Query / Question” (which could itself be text, an image, or a combination) arrives, it’s also processed by the “Multimodal Embedding Model.”

The resulting query embedding is used for “Multimodal Retrieval” from the vector database to find the most relevant multimodal data chunks. This retrieved data, along with the original query, undergoes “Prompt Processing” and is then passed to a “Large Vision Language Model Inference” (or a general multimodal LLM) to synthesize a “Response / Answer” that can integrate information from the diverse retrieved sources.

The significance of Multimodal RAG lies in its ability to answer more complex and nuanced queries that require understanding and correlating information across different formats.

For example, a user could ask, “Show me images where our product appears in an outdoor setting” or “Summarize the key points from the video segment where the new feature is demonstrated.” By bridging the gap between different data types, Multimodal RAG enables a richer and more comprehensive interaction with diverse knowledge bases.

---

## 4. Graph RAG: Uses Knowledge Graphs for Context

Graph RAG enhances traditional Retrieval Augmented Generation by incorporating structured knowledge from Knowledge Graphs (KGs) alongside unstructured data from vector databases.

Knowledge Graphs represent information as entities (nodes) and relationships (edges), allowing for a more nuanced and interconnected understanding of data.

This structure enables the RAG system to retrieve not just semantically similar text chunks but also related entities and their explicit connections, leading to more precise and contextually rich answers, especially for queries that involve complex relationships or multi-hop reasoning.

As illustrated in the diagram, when a “User” poses a “Question,” a “Graph Extraction” step may occur. This step can involve identifying key entities and relationships mentioned in the question, which are then used to query the Knowledge Graph.

Simultaneously or subsequently, the “Retrieval” process accesses both the “Knowledge Graphs” and a “Vector Database.” The “Retrieved Context” from these sources- facts and relationships from the KG and relevant text passages from the vector store — is then combined.

This enriched context, along with the original “Question,” forms the basis for “Prompt Augmentation,” which is then fed to an LLM to generate the final “Answer.”

The key advantage of Graph RAG is its ability to leverage the explicit relational information stored in KGs. While vector databases excel at finding semantically similar text, KGs can provide direct answers to questions like “Who is the CEO of Company X and what other companies are they affiliated with?” or “What are the common side effects of Drug Y and how do they interact with Condition Z?”.

By combining the strengths of both graph-based retrieval and semantic vector search, Graph RAG systems can provide more accurate, comprehensive, and interpretable answers.


In [ ]:
# Conceptual Python for Graph RAG
# We'll mock a simple KG and Vector DB

# --- Mock Knowledge Graph Data (e.g., using dictionaries to represent nodes/edges) ---
# Nodes: {id: {properties}}
# Edges: [(source_id, relation, target_id, {properties})]
mock_kg_nodes = {
    "company_A": {"name": "Innovate Corp", "type": "company", "industry": "Tech"},
    "person_X": {"name": "Alice Wonderland", "type": "person", "role": "CEO"},
    "product_P1": {"name": "Synergy AI", "type": "product", "launched": 2023},
    "feature_F1": {"name": "Real-time Analytics", "type": "feature"}
}
mock_kg_edges = [
    ("person_X", "CEO_OF", "company_A", {}),
    ("company_A", "DEVELOPS", "product_P1", {}),
    ("product_P1", "HAS_FEATURE", "feature_F1", {})
]

# --- Mock Vector Database ---
mock_vector_db_chunks = [
    {"id": "doc1", "text": "Innovate Corp announced impressive Q4 earnings, largely driven by their flagship product, Synergy AI."},
    {"id": "doc2", "text": "Synergy AI's real-time analytics feature has been praised by early adopters for its speed and accuracy."},
    {"id": "doc3", "text": "Alice Wonderland, CEO of Innovate Corp, presented the future vision at the annual shareholder meeting."},
    {"id": "doc4", "text": "The tech industry is rapidly evolving, with AI being a major disruptor."}
]

# --- Graph Extraction (Simplified) ---
def extract_entities_from_question(question: str) -> list[str]:
    """Simplistic entity extraction based on keywords."""
    entities = []
    question_lower = question.lower()
    if "innovate corp" in question_lower:
        entities.append("company_A")
    if "alice wonderland" in question_lower:
        entities.append("person_X")
    if "synergy ai" in question_lower:
        entities.append("product_P1")
    # In a real system, this would use NLP/NER models.
    print(f"GRAPH EXTRACTION: Extracted entities from question: {entities}")
    return entities

# --- Retrieval Functions ---
def query_knowledge_graph(entities: list[str]) -> list[str]:
    """
    Simulates querying a knowledge graph.
    For each entity, find its direct relations and related entities.
    """
    kg_context = []
    if not entities:
        return kg_context

    for entity_id in entities:
        if entity_id in mock_kg_nodes:
            node_info = mock_kg_nodes[entity_id]
            kg_context.append(f"Entity: {node_info.get('name', entity_id)} (Type: {node_info.get('type', 'N/A')})")
            for source, relation, target, props in mock_kg_edges:
                if source == entity_id:
                    target_name = mock_kg_nodes.get(target, {}).get('name', target)
                    kg_context.append(f"  - {relation} -> {target_name}")
                elif target == entity_id:
                    source_name = mock_kg_nodes.get(source, {}).get('name', source)
                    # Inferring reverse relation for simplicity
                    kg_context.append(f"  - IS_{relation}_OF -> {source_name}")
    print(f"KG RETRIEVAL: Retrieved from KG: {kg_context}")
    return kg_context

def query_vector_database(question: str, top_k: int = 2) -> list[str]:
    """Simulates semantic search in a vector database."""
    # Very basic keyword matching for simulation
    results = []
    question_tokens = set(question.lower().split())
    for chunk in mock_vector_db_chunks:
        chunk_tokens = set(chunk["text"].lower().split())
        if question_tokens.intersection(chunk_tokens): # Simple check
            results.append(chunk["text"])
    print(f"VECTOR DB RETRIEVAL: Retrieved from Vector DB: {results[:top_k]}")
    return results[:top_k]

# --- LLM for Generation (Simplified) ---
def llm_generate_response(question: str, context: str) -> str:
    # Simulates the LLM generating a response based on query and augmented context
    return f"Question: {question}\nAugmented Context:\n{context}\n\nAnswer: Based on the provided information, [LLM synthesized answer here based on context and question]."

# --- GraphRAG Pipeline ---
def graph_rag_pipeline(user_question: str):
    print(f"\nUSER QUESTION: {user_question}")

    # 1. Graph Extraction (from question)
    extracted_entities = extract_entities_from_question(user_question)

    # 2. Retrieval
    # 2a. From Knowledge Graph
    kg_retrieved_context = query_knowledge_graph(extracted_entities)
    # 2b. From Vector Database
    vector_db_retrieved_context = query_vector_database(user_question)

    # 3. Combine Retrieved Context
    combined_context_list = kg_retrieved_context + vector_db_retrieved_context
    retrieved_context_str = "\n".join(combined_context_list)
    print(f"COMBINED CONTEXT:\n{retrieved_context_str}")

    # 4. Prompt Augmentation & Generation
    # The augmented prompt is implicitly created by passing question and context to LLM
    final_answer = llm_generate_response(user_question, retrieved_context_str)
    print(f"FINAL ANSWER:\n{final_answer}")
    return final_answer

# --- Example Usage ---
graph_rag_pipeline("Tell me about Innovate Corp and its CEO.")
print("-" * 50)
graph_rag_pipeline("What are the features of Synergy AI?")
print("-" * 50)
graph_rag_pipeline("Who is Alice Wonderland and what products are related to her company?")


This conceptual code demonstrates how information from a mock Knowledge Graph (entities and relationships) and a mock Vector Database (text chunks) can be retrieved and combined.

The “Graph Extraction” step is simplified to keyword spotting. In a real system, this would involve more sophisticated Natural Language Processing (NLP) techniques for entity recognition and relation extraction to formulate precise graph queries (e.g., using Cypher for Neo4j, SPARQL for RDF graphs).

The retrieved graph data provides structured facts, while the vector DB provides relevant textual context, and together they form a richer input for the LLM.

---

## 5. Hybrid RAG: Blends Retrieval Techniques

Hybrid RAG combines the strengths of dense (semantic/vector) and sparse (symbolic/keyword or graph-based) retrieval methods to boost recall and relevance in Retrieval-Augmented Generation systems.

1. Enterprise Data Ingestion: Data from various sources (text, tables, images, etc.) is parsed and prepared for indexing using both vector-based and symbolic pipelines.
2. Dual Indexing

- A Vector Database is created using embedding models (e.g., OpenAI, Sentence-BERT, CLIP for images).
- A Symbolic Index is built using keyword-based or graph-based methods (BM25, Elasticsearch, or a Knowledge Graph).

3. User Query: A user submits a query (e.g., “Show me reports about failed deliveries in Q4”).

4. Parallel Retrieval

- The query is embedded and searched in the vector database (semantic similarity).
- Simultaneously, symbolic retrieval searches for exact matches or relations (e.g., “failed deliveries” AND “Q4”).

5. Result Fusion: Results from both sources are merged, ranked, or reranked based on relevance scores, metadata, or a trained fusion model.

6. Prompt Construction: The combined relevant information is formatted into a structured prompt for the language model.

7. LLM Inference: The prompt (containing both semantically and symbolically retrieved context) is passed to a large language model for grounded generation.

8. Response Generation: The LLM produces an answer that is contextually rich, grounded in both unstructured meaning and structured facts.

---

## 6. Agentic (Router) RAG: Agent Decides Retrieval Strategy

Agentic (Router) RAG introduces an intelligent agent, typically powered by a Large Language Model (LLM), that acts as a decision-maker or “router” before any data retrieval occurs.

Instead of a fixed retrieval pipeline, this agent analyzes the incoming query and determines the most appropriate data source or retrieval method. For instance, a query about recent news might be routed to a web search tool, a question about internal company policy to a specific document vector store, and a factual question about product specifications to another.

This dynamic routing allows the RAG system to tap into diverse and specialized knowledge bases, optimizing for relevance and accuracy.

The core mechanism involves the agent assessing the query’s intent, keywords, or semantic meaning. Based on this assessment, it selects one or more “tools” or data sources from an available set.

As shown in the diagram, these tools could be different vector search engines (e.g., “Vector search engine A” querying “Collection A”, “Vector search engine B” querying “Collection B”), a traditional web search, a calculator, or even APIs to other services.

Once the agent selects the appropriate tool(s), the query is processed by that tool, relevant context is retrieved, and then passed to a generator LLM (which could be the same LLM as the agent or a different one) to synthesize the final response.

This approach significantly enhances the RAG system’s versatility and efficiency. By routing queries to the most relevant sources, it reduces noise from irrelevant data, improves the quality of retrieved context, and can handle a much broader range of query types.

It’s particularly useful when dealing with multiple, distinct knowledge domains or when different types of queries require fundamentally different retrieval strategies (e.g., semantic search vs. keyword search vs. structured data lookup).

Here is a conceptual code that illustrates how an agent might direct a query.


In [ ]:
# Conceptual Python using a simplified agent logic
# In a real scenario, the "agent_decide_route" would involve an LLM call.

# --- Mock Tools/Data Sources ---
def search_vector_db_A(query: str) -> str:
    # Simulates searching a specific vector database (e.g., product documentation)
    print(f"ROUTER: Querying Vector DB A for: '{query}'")
    if "product specs for X" in query.lower():
        return "Collection A: Product X has 16GB RAM, 512GB SSD, and a 15-inch display."
    return "Collection A: No specific information found."

def search_vector_db_B(query: str) -> str:
    # Simulates searching another vector database (e.g., HR policies)
    print(f"ROUTER: Querying Vector DB B for: '{query}'")
    if "company holiday policy" in query.lower():
        return "Collection B: Employees are entitled to 25 paid holidays per year."
    return "Collection B: No specific information found."

def web_search_tool(query: str) -> str:
    # Simulates a web search
    print(f"ROUTER: Performing web search for: '{query}'")
    if "latest news on AI" in query.lower():
        return "Web Search: Major advancements in generative AI reported this week."
    return "Web Search: No specific, current news found."

def calculator_tool(expression: str) -> str:
    # Simulates a calculator tool
    print(f"ROUTER: Calculating: '{expression}'")
    try:
        # A very basic calculator for demonstration
        if "what is" in expression.lower():
            expression_to_eval = expression.lower().replace("what is", "").strip().replace("?", "")
            return f"Calculator: The result of {expression_to_eval} is {eval(expression_to_eval)}."
        return f"Calculator: Could not understand expression '{expression}'"
    except Exception as e:
        return f"Calculator: Error evaluating expression: {str(e)}"

# --- Agent Logic (Simplified) ---
def agent_decide_route(query: str) -> tuple[callable, str]:
    """
    This is a simplified router. A real agent would use an LLM
    with tool descriptions to decide the best route.
    """
    query_lower = query.lower()
    if "product specs" in query_lower:
        return search_vector_db_A, query
    elif "policy" in query_lower or "hr" in query_lower:
        return search_vector_db_B, query
    elif "news" in query_lower or "current events" in query_lower:
        return web_search_tool, query
    elif "calculate" in query_lower or "what is" in query_lower and any(c in query_lower for c in "+-*/"):
        # Pass the relevant part to the calculator
        return calculator_tool, query # The calculator tool itself might do further parsing
    else:
        # Default route or could raise an error / ask for clarification
        print("ROUTER: No specific tool matched, defaulting to Vector DB A.")
        return search_vector_db_A, query

# --- LLM for Generation (Simplified) ---
def llm_generate_response(query: str, context: str) -> str:
    # Simulates the LLM generating a response based on query and context
    return f"Query: {query}\nAnswer: Based on the retrieved information, {context}"

# --- Main Agentic (Router) RAG Flow ---
def agentic_router_rag_pipeline(user_query: str):
    print(f"\nUSER QUERY: {user_query}")

    # 1. Agent decides the route/tool
    tool_function, tool_input_query = agent_decide_route(user_query)

    # 2. Execute the chosen tool to retrieve context
    retrieved_context = tool_function(tool_input_query)
    print(f"RETRIEVED CONTEXT: {retrieved_context}")

    # 3. LLM generates the final response
    final_response = llm_generate_response(user_query, retrieved_context)
    print(f"FINAL RESPONSE:\n{final_response}")
    return final_response

# --- Example Usage ---
agentic_router_rag_pipeline("What are the product specs for X?")
agentic_router_rag_pipeline("What is the company holiday policy?")
agentic_router_rag_pipeline("What is the latest news on AI?")
agentic_router_rag_pipeline("Calculate what is 5 * (10 + 3)?")
agentic_router_rag_pipeline("Tell me a story.") # Example of a query that might default


In a real-world LangChain or LlamaIndex implementation, the agent_decide_route function would involve an LLM call, where the LLM is prompted with the query and descriptions of available tools (like Vector search engine A, Web search, Calculator), and it would output which tool to use and what input to pass to it. The chosen tool would then be executed.

---

## 7. Agentic (Multi-Agent) RAG: Multiple Agents Collaborate using Different Tools

Agentic (Multi-Agent) RAG takes the concept of specialized agents a step further by creating a system where multiple autonomous agents collaborate to fulfill a complex query.

Instead of a single router agent, you might have a primary agent that can decompose a query or task and delegate sub-tasks to other specialized agents.

Each of these agents can have its own distinct set of tools, knowledge bases, and even reasoning capabilities. This architecture allows for a sophisticated division of labor, where complex problems are broken down into manageable parts, tackled by the most suitable agent, and the results are then synthesized.

As depicted in the diagram, a user’s “Query” first goes to an initial “Retrieval Agent.” This agent might act as a coordinator or a primary reasoner.

It can then communicate or delegate tasks to other specialized agents:

- “Retrieval Agent A” (which might focus on structured data from “Collection A” and “Collection B” via their respective vector search engines)
- “Retrieval Agent B” (which could be an expert in “Web search” for current events or broad knowledge)
- “Retrieval Agent C” (which might be tasked with searching internal communication platforms like “Slack” or “Gmail”).

Each of these sub-agents performs its task using its dedicated tools, and their findings are then passed back, potentially to the initial agent or a designated aggregator, before being fed to the final “LLM” for synthesizing the “Response.”

The power of this multi-agent system lies in its ability to handle highly multifaceted queries that require information from disparate sources and potentially different types of reasoning.

For example, a query like “Summarize the latest client feedback from Gmail regarding Project X, cross-reference it with internal discussions on Slack, and compare it against the project goals outlined in Collection A” would be very challenging for a simpler RAG system.

A multi-agent RAG can assign parts of this query to agents specializing in Gmail, Slack, and internal document retrieval, respectively. The collaboration might involve agents passing information to each other or the coordinating agent gathering all pieces to form a comprehensive context for the final LLM.
